# LLaVA-OneVision: 一个模型处理单图、多图、视频

在LLaVA-OneVision之前，开源VLM有不同的派系：LLaVA-1.5 做单图，Mantis和VILA做多图，Video-LLaVA和Video-LLaMA做视频。每个都在自己的基准上胜利，但是在其他方向失败。LLaVA-OneVision 训练一个模型，处理上面三种场景，并且涌现了任务迁移效应（单图技能输出到视频，多图推理输出到单图）胜过各路专家之和。配方极其简单：一个跨场景保持恒定的视觉Token预算，加上一套从单图到OneVision多图再到视频的训练过程。

## 问题描述

单图、多图以及视频各自以不同方式给模型施压。

单图要的是高分辨率（AnyRes 大约2880个视觉token）来抓OCR和细节。预算是：每张图，2880个token。

多图要的是几个钟分辨率的图（每个大约576个视觉token）来支持图像间推理。预算是：4～8张图，每张图576个token，2300～4600个token。

视频要的则是低分辨率的许多帧（池化后大约196个token）来捕捉时许动态。预算是：8～32帧，每帧196个token，1600～6200个token。

如果你训练单独的模型，就是选择一种预算。如果你训练一个模型，你需要预算在三种场景之间合理缩放，不至于撑爆上下文。

在OneVision出现之前，回答是“训练一个场景，忽略其他场景”。Video-LLaVA靠额外的训练将视频后装到一个图像模型上。LLaVA-NeXT用切块加了多图支持。没有哪个能够干净地搞定全部三个。

## 基本概念

### OneVision的token预算

LLaVA-OneVision 对每个样本选择了统一的大约3000～4000token的预算，根据场景分为：
- 单图。 AnyRes-9（3x3 瓦片+缩略图），每个瓦片在384分辨率下有729个patch，通过激进的双线性池化池后，每个瓦片182个patch。总数就是182x9+182共1820个token。 或者直接AnyRes-4，每个瓦片729个patch，729x4+729 大约3720个token。
- 多图。每张图分辨率384，不带池化分成729个patches，也就是729个token。预算是六张图大约4374个token。
- 视频。分辨率384，帧率32，使用3x3双线性池化，即每帧81个token。总数大约2592个token。

编码器不同场景产出不同的形状，但是LLM始终消费同等预算的视觉token。

### 训练步骤

LLaVA-OneVision 使用三阶段训练。
- 单图SFT（即Single-Image SFT ，SI）。  所有的数据是单图加上文本。在高分辨率的AnyRes输入上训练，这教会模型感知、OCR以及细节理解。
- OneVision SFT（OV）。  混合单图、多图和视频。在统一的token预算上训练，这教会模型处理不同的批形状。不需要重制权重，从SI接着训练。
- 任务迁移（Task Transfoer， TT） 继续在一个目标任务配比上训练，通常按产品偏向多图或者视频。也就是可选的部署微调。

关键点：训练顺序很重要。先训多图或者视频比先训单图表现更差。

### 为什么有效

单图训练打下感知基础。Patch tokens 携带细粒度的视觉特征。LLM学会把它们与文档整合。多图和视频引入了结构化的挑战（那张图对应哪些，哪些事情先发生），在没有强的感知基础的情况下，很难学到。

如果你从一开始就训三种场景，模型的感知很差（每批次中单图数据不过），并且对结构过拟合（许多多图/视频数据）。导致：一个会遵循跨图推理模式但是视觉很浅的模型。

所以SI阶段给你感知强度，OV阶段在不损失单图能力的情况下给了组合/时序上的推理。

### 涌现的跨场景技能

LLaVA-OneVision 论文报告了三种涌现的能力。
- 多摄像头推理。  通过在多图和视频的训练，推理阶段让它基于多视角的驾驶场景做视觉推理。
- 标注集提示词。   用户在一张图中使用需要标注，然后模型可以推理出标记3相对于标记7在做什么。
- Iphone截图agent。  用户提供一张IPhone的屏幕截图，然后让决定下一个电机位置。

### 视觉Token池化

Token预算要求进行池化，OneVision使用的是在2D Patch网格上的双线性插值，24x24=576个patches 变成 12x12=144个patch（2倍）或者8x8=64个patch（3倍）。池化在patch网格空间内完成，而不是token空间，为了保持局部性。

每个场景的池化倍率是一个超参数。池化得越少，视觉token越多，带来越强的表征能力。池化得越多，视觉token越少，塞得下更多帧和图像。


# 开始编码

教学积木：LLaVA-OneVision 核心——**统一视觉 token 预算**、**单图 AnyRes+池化 / 多图 / 视频** 三条编码路径、**2D 网格双线性池化**、**SI→OV 训练阶段切换**。


## 1. 配置 + 2D Patch 网格双线性池化

池化在空间网格上做，而不是把 token 当一维序列随便抽稀，以保持局部性。


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from enum import Enum
from typing import Literal

import torch
import torch.nn as nn
import torch.nn.functional as F


class Modality(str, Enum):
    """OneVision 支持的三种输入场景。"""

    SINGLE = "single"
    MULTI = "multi"
    VIDEO = "video"


class TrainStage(str, Enum):
    """训练阶段：先 SI（单图感知）再 OV（混合场景）。"""

    SI = "si"
    OV = "ov"


@dataclass
class TinyOneVisionConfig:
    """小尺寸 OneVision 示意配置（非真实 384/729）。"""

    patch_size: int = 8
    """玩具 patch 边长。"""

    encoder_size: int = 32
    """固定编码器输入边长（真模型常 384）。"""

    vit_dim: int = 32
    """每个视觉 token 维。"""

    llm_dim: int = 64
    """投影到 LLM 的维。"""

    # --- 各场景池化后的目标网格（决定每块/每帧 token 数）---
    single_pool_hw: tuple[int, int] = (2, 2)
    """单图：每个瓦片/缩略图池化后的 (H,W)，token=H*W。"""

    multi_pool_hw: tuple[int, int] = (4, 4)
    """多图：每张图池化后网格（更少池化=更高清，图数更少）。"""

    video_pool_hw: tuple[int, int] = (2, 2)
    """视频：每帧更激进池化，腾出帧数预算。"""

    anyres_grids: tuple[tuple[int, int], ...] = ((1, 1), (1, 2), (2, 1), (2, 2))
    """单图 AnyRes 候选瓦片网格。"""

    token_budget: int = 48
    """统一预算上限（玩具数；真模型约 3000~4000）。"""

    @property
    def tokens_per_encoder_image(self) -> int:
        """固定编码器、未池化时一张方图的 patch 数。"""
        s = self.encoder_size // self.patch_size
        return s * s


def resize_image(image: torch.Tensor, size_h: int, size_w: int) -> torch.Tensor:
    """
    Args:
        image: ``(3, H, W)``。
        size_h: 目标高。
        size_w: 目标宽。

    Returns:
        resized: ``(3, size_h, size_w)``。
    """
    x = F.interpolate(
        image.unsqueeze(0),
        size=(size_h, size_w),
        mode="bilinear",
        align_corners=False,
    )
    return x.squeeze(0)


def bilinear_pool_token_grid(
    tokens: torch.Tensor,
    grid_h: int,
    grid_w: int,
    out_h: int,
    out_w: int,
) -> torch.Tensor:
    """
    在 2D patch 网格上对 token 做双线性下采样（OneVision 式视觉 token 池化）。

    Args:
        tokens: ``(N, D)``，且 ``N == grid_h * grid_w``。
        grid_h: 当前网格高。
        grid_w: 当前网格宽。
        out_h: 池化后网格高。
        out_w: 池化后网格宽。

    Returns:
        pooled: ``(out_h * out_w, D)``。
    """
    if tokens.ndim != 2:
        raise ValueError(f"expect (N,D), got {tuple(tokens.shape)}")
    n, d = tokens.shape
    if n != grid_h * grid_w:
        raise ValueError(f"N={n} != grid_h*grid_w={grid_h * grid_w}")
    # (1, D, Gh, Gw) 才能对空间维插值
    x = tokens.transpose(0, 1).reshape(1, d, grid_h, grid_w)
    x = F.interpolate(x, size=(out_h, out_w), mode="bilinear", align_corners=False)
    return x.reshape(d, out_h * out_w).transpose(0, 1).contiguous()


print("config + bilinear_pool_token_grid ready")


## 2. 固定编码器 + 三条场景编码（单图 AnyRes / 多图 / 视频）


In [ ]:
class PatchEmbed(nn.Module):
    """把 ``encoder_size`` 方图切成 patch 并线性嵌入。"""

    def __init__(self, cfg: TinyOneVisionConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.side = cfg.encoder_size // cfg.patch_size
        self.proj = nn.Linear(3 * cfg.patch_size * cfg.patch_size, cfg.vit_dim)

    def forward(self, image: torch.Tensor) -> torch.Tensor:
        """
        Args:
            image: ``(3, encoder_size, encoder_size)``。

        Returns:
            tokens: ``(side*side, vit_dim)``。
        """
        p = self.cfg.patch_size
        _, h, w = image.shape
        if (h, w) != (self.cfg.encoder_size, self.cfg.encoder_size):
            raise ValueError(f"expect square {self.cfg.encoder_size}, got {(h, w)}")
        gh = gw = self.side
        x = image.reshape(3, gh, p, gw, p).permute(1, 3, 0, 2, 4).reshape(gh * gw, -1)
        return self.proj(x)


class FixedVisionEncoder(nn.Module):
    """冻结固定分辨率视觉编码器示意。"""

    def __init__(self, cfg: TinyOneVisionConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.embed = PatchEmbed(cfg)
        for p in self.parameters():
            p.requires_grad = False

    def encode_square(self, image: torch.Tensor) -> torch.Tensor:
        """
        Args:
            image: ``(3, H, W)``，内部先缩放到 ``encoder_size``。

        Returns:
            tokens: ``(S*S, vit_dim)`` 未池化。
        """
        sq = resize_image(image, self.cfg.encoder_size, self.cfg.encoder_size)
        return self.embed(sq)


def select_anyres_grid(
    height: int,
    width: int,
    candidates: tuple[tuple[int, int], ...],
) -> tuple[int, int]:
    """
    Args:
        height: 原图高。
        width: 原图宽。
        candidates: ``(gh, gw)`` 候选。

    Returns:
        best: 与长宽比最接近的网格。
    """
    aspect = width / max(height, 1)
    best = candidates[0]
    best_score = float("inf")
    for gh, gw in candidates:
        score = abs((gw / gh) - aspect) + 1e-3 * (gh * gw)
        if score < best_score:
            best_score = score
            best = (gh, gw)
    return best


def encode_single_image(
    image: torch.Tensor,
    encoder: FixedVisionEncoder,
    cfg: TinyOneVisionConfig,
) -> torch.Tensor:
    """
    单图路径：AnyRes（缩略图 + 瓦片）→ 每块 2D 池化 → 拼接。

    Args:
        image: ``(3, H, W)``。
        encoder: 固定分辨率编码器。
        cfg: 配置（使用 ``single_pool_hw`` / ``anyres_grids``）。

    Returns:
        tokens: ``((1 + gh*gw) * out_h * out_w, vit_dim)``。
    """
    _, H, W = image.shape
    gh, gw = select_anyres_grid(H, W, cfg.anyres_grids)
    S = cfg.encoder_size
    out_h, out_w = cfg.single_pool_hw
    side = encoder.embed.side

    def _encode_and_pool(crop: torch.Tensor) -> torch.Tensor:
        raw = encoder.encode_square(crop)
        return bilinear_pool_token_grid(raw, side, side, out_h, out_w)

    thumb = _encode_and_pool(image)
    canvas = resize_image(image, gh * S, gw * S)
    parts: list[torch.Tensor] = [thumb]
    for i in range(gh):
        for j in range(gw):
            tile = canvas[:, i * S : (i + 1) * S, j * S : (j + 1) * S]
            parts.append(_encode_and_pool(tile))
    return torch.cat(parts, dim=0)


def encode_multi_images(
    images: list[torch.Tensor],
    encoder: FixedVisionEncoder,
    cfg: TinyOneVisionConfig,
) -> torch.Tensor:
    """
    多图路径：每张图中等分辨率编码 + 较轻池化，再在序列维拼接。

    Args:
        images: 长度 ``T`` 的 ``(3, H_i, W_i)`` 列表。
        encoder: 固定分辨率编码器。
        cfg: 使用 ``multi_pool_hw``。

    Returns:
        tokens: ``(T * out_h * out_w, vit_dim)``。
    """
    out_h, out_w = cfg.multi_pool_hw
    side = encoder.embed.side
    parts: list[torch.Tensor] = []
    for img in images:
        raw = encoder.encode_square(img)
        parts.append(bilinear_pool_token_grid(raw, side, side, out_h, out_w))
    return torch.cat(parts, dim=0)


def encode_video(
    frames: torch.Tensor,
    encoder: FixedVisionEncoder,
    cfg: TinyOneVisionConfig,
) -> torch.Tensor:
    """
    视频路径：多帧 + 更强池化，控制总 token。

    Args:
        frames: ``(T, 3, H, W)``。
        encoder: 固定分辨率编码器。
        cfg: 使用 ``video_pool_hw``。

    Returns:
        tokens: ``(T * out_h * out_w, vit_dim)``。
    """
    if frames.ndim != 4:
        raise ValueError(f"expect (T,3,H,W), got {tuple(frames.shape)}")
    out_h, out_w = cfg.video_pool_hw
    side = encoder.embed.side
    parts: list[torch.Tensor] = []
    for t in range(frames.size(0)):
        raw = encoder.encode_square(frames[t])
        parts.append(bilinear_pool_token_grid(raw, side, side, out_h, out_w))
    return torch.cat(parts, dim=0)


print("scene encoders ready")


## 3. 统一预算裁剪 + MLP 投影 + TinyOneVision 前端子 + SI/OV 阶段


In [ ]:
class MLPProjector(nn.Module):
    """两层 MLP：``vit_dim -> llm_dim``（LLaVA 风格）。"""

    def __init__(self, vit_dim: int, llm_dim: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(vit_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim),
        )

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        """
        Args:
            tokens: ``(N, vit_dim)``。

        Returns:
            projected: ``(N, llm_dim)``。
        """
        return self.net(tokens)


def fit_token_budget(
    tokens: torch.Tensor,
    budget: int,
) -> torch.Tensor:
    """
    将视觉 token 数压进统一预算：超出则均匀下采样，不足则原样返回。

    Args:
        tokens: ``(N, D)``。
        budget: 最大 token 数。

    Returns:
        fitted: ``(N', D)``，``N' <= budget``。
    """
    n = tokens.size(0)
    if n <= budget:
        return tokens
    # 在 token 维均匀取 budget 个下标（教学简化；真系统靠场景池化控预算）
    idx = torch.linspace(0, n - 1, budget, device=tokens.device).round().long()
    return tokens.index_select(0, idx)


class TinyOneVisionFrontEnd(nn.Module):
    """
    OneVision 视觉前端：按 modality 选编码路径，统一预算后投影到 LLM 维。
    """

    def __init__(self, cfg: TinyOneVisionConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.encoder = FixedVisionEncoder(cfg)
        self.projector = MLPProjector(cfg.vit_dim, cfg.llm_dim)
        self.stage: TrainStage = TrainStage.SI

    def set_stage(self, stage: TrainStage) -> None:
        """
        Args:
            stage: ``SI`` 时建议只喂单图；``OV`` 允许三种 modality。
        """
        self.stage = stage

    def forward_visual(
        self,
        modality: Modality,
        image: torch.Tensor | None = None,
        images: list[torch.Tensor] | None = None,
        frames: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """
        Args:
            modality: 场景类型。
            image: 单图 ``(3,H,W)``（``SINGLE`` 必填）。
            images: 多图列表（``MULTI`` 必填）。
            frames: 视频 ``(T,3,H,W)``（``VIDEO`` 必填）。

        Returns:
            llm_tokens: ``(N', llm_dim)``，``N' <= token_budget``。

        Raises:
            ValueError: 阶段与场景不匹配，或缺少对应输入。
        """
        if self.stage == TrainStage.SI and modality != Modality.SINGLE:
            raise ValueError("SI 阶段只接受 SINGLE 模态（先打感知基础）")

        if modality == Modality.SINGLE:
            if image is None:
                raise ValueError("SINGLE 需要 image")
            vis = encode_single_image(image, self.encoder, self.cfg)
        elif modality == Modality.MULTI:
            if not images:
                raise ValueError("MULTI 需要非空 images")
            vis = encode_multi_images(images, self.encoder, self.cfg)
        elif modality == Modality.VIDEO:
            if frames is None:
                raise ValueError("VIDEO 需要 frames")
            vis = encode_video(frames, self.encoder, self.cfg)
        else:
            raise ValueError(modality)

        vis = fit_token_budget(vis, self.cfg.token_budget)
        return self.projector(vis)

    def estimate_raw_tokens(
        self,
        modality: Modality,
        *,
        grid: tuple[int, int] | None = None,
        num_images: int | None = None,
        num_frames: int | None = None,
    ) -> int:
        """
        估算池化后、预算裁剪前的 token 数（便于对照笔记里的预算表）。

        Args:
            modality: 场景。
            grid: 单图 AnyRes ``(gh,gw)``；默认用 ``(2,2)`` 示意。
            num_images: 多图张数。
            num_frames: 视频帧数。

        Returns:
            n: 理论 token 数。
        """
        if modality == Modality.SINGLE:
            gh, gw = grid or (2, 2)
            oh, ow = self.cfg.single_pool_hw
            return (1 + gh * gw) * oh * ow
        if modality == Modality.MULTI:
            t = num_images or 1
            oh, ow = self.cfg.multi_pool_hw
            return t * oh * ow
        if modality == Modality.VIDEO:
            t = num_frames or 1
            oh, ow = self.cfg.video_pool_hw
            return t * oh * ow
        raise ValueError(modality)


print("TinyOneVisionFrontEnd ready")


## 4. 冒烟测试：三场景 token 数、预算、SI 约束、投影 shape


In [ ]:
def smoke_test() -> None:
    """验证 OneVision 教学前端的核心不变量。"""
    torch.manual_seed(0)
    cfg = TinyOneVisionConfig(token_budget=40)
    fe = TinyOneVisionFrontEnd(cfg)

    print("=== theoretical budgets (before fit_token_budget) ===")
    print("single AnyRes-2x2:", fe.estimate_raw_tokens(Modality.SINGLE, grid=(2, 2)))
    print("multi 3 images   :", fe.estimate_raw_tokens(Modality.MULTI, num_images=3))
    print("video 8 frames   :", fe.estimate_raw_tokens(Modality.VIDEO, num_frames=8))

    # --- SI: only single ---
    fe.set_stage(TrainStage.SI)
    single = torch.randn(3, 48, 32)
    tok_s = fe.forward_visual(Modality.SINGLE, image=single)
    print("\n=== SI single ===")
    print(f"llm visual tokens: {tuple(tok_s.shape)}  # (N, llm_dim), N<=budget")
    assert tok_s.size(0) <= cfg.token_budget
    assert tok_s.size(1) == cfg.llm_dim

    try:
        fe.forward_visual(Modality.MULTI, images=[single, single])
        raise AssertionError("SI should reject MULTI")
    except ValueError as e:
        print(f"SI rejects multi: {e}")

    # --- OV: all modalities under same budget ---
    fe.set_stage(TrainStage.OV)
    imgs = [torch.randn(3, 40, 40), torch.randn(3, 32, 48), torch.randn(3, 48, 32)]
    tok_m = fe.forward_visual(Modality.MULTI, images=imgs)
    frames = torch.randn(8, 3, 32, 32)
    tok_v = fe.forward_visual(Modality.VIDEO, frames=frames)
    tok_s2 = fe.forward_visual(Modality.SINGLE, image=single)

    print("\n=== OV shapes (unified budget) ===")
    print(f"single: {tuple(tok_s2.shape)}")
    print(f"multi : {tuple(tok_m.shape)}")
    print(f"video : {tuple(tok_v.shape)}")
    assert tok_m.size(0) <= cfg.token_budget
    assert tok_v.size(0) <= cfg.token_budget

    # 2D pool locality smoke: grid interpolate reduces N
    raw = torch.randn(cfg.tokens_per_encoder_image, cfg.vit_dim)
    side = int(cfg.tokens_per_encoder_image**0.5)
    pooled = bilinear_pool_token_grid(raw, side, side, 2, 2)
    assert pooled.shape == (4, cfg.vit_dim)
    print(f"\npool {side}x{side} -> 2x2 : {tuple(pooled.shape)}")
    print("SMOKE TEST OK")


smoke_test()
